In [ ]:
from sklearn.model_selection import train_test_split
from rich import print
from pathlib import Path
import shutil


def create_split_folders(folder_name, images, masks):
    Path(f"{folder_name}/images").mkdir(parents=True, exist_ok=True)
    Path(f"{folder_name}/masks").mkdir(parents=True, exist_ok=True)

    for img, msk in zip(images, masks):
        destination_img = Path(folder_name) / "images" / img.name
        destination_msk = Path(folder_name) / "masks" / msk.name
        shutil.move(img, destination_img)
        shutil.move(msk, destination_msk)


def split_images_and_masks(
    image_glob, mask_glob, train_size=0.8, test_size=0.2, random_state=42
):
    image_paths = sorted(list(Path(image_glob).glob("*.tif")))
    mask_paths = sorted(list(Path(mask_glob).glob("*.tif")))

    train_images, dev_images, train_masks, dev_masks = train_test_split(
        image_paths, mask_paths, train_size=train_size, random_state=random_state
    )

    val_images, test_images, val_masks, test_masks = train_test_split(
        dev_images, dev_masks, test_size=test_size, random_state=random_state
    )

    return train_images, val_images, test_images, train_masks, val_masks, test_masks

In [ ]:
DATASET_NAME = "Vaihingen"
IMAGE_FOLDER = "top"
MASK_FOLDER = "labels"

IMAGE_GLOB = f"{DATASET_NAME}/{IMAGE_FOLDER}"
MASK_GLOB = f"{DATASET_NAME}/{MASK_FOLDER}"
TRAIN_FOLDER = f"{DATASET_NAME}/train"
VAL_FOLDER = f"{DATASET_NAME}/val"
TEST_FOLDER = f"{DATASET_NAME}/test"


In [ ]:
train_size = 0.8
test_size = 0.2
random_state = 42


train_images, val_images, test_images, train_masks, val_masks, test_masks = (
    split_images_and_masks(
        IMAGE_GLOB, MASK_GLOB, test_size=test_size, random_state=random_state
    )
)

print(
    f"[bold yellow] The process with create {len(train_images)} training images, {len(val_images)} validation images, and {len(test_images)} test images.[/bold yellow]"
)

output = input("You want to proceed with moving the files? (y/n): ")

if output == "y":
    print("[bold blue] Proceeding with file move... [/bold blue]")
    # create_split_folders(TRAIN_FOLDER, images=train_images, masks=train_masks)
    # create_split_folders(VAL_FOLDER, images=val_images, masks=val_masks)
    # create_split_folders(TEST_FOLDER, images=test_images, masks=test_masks)
    print("[bold green] Dataset succesfully splitted! [/bold green]")
else:
    print("[bold red] Aborting file move. [/bold red]")

 The process with create 26 training images, 5 validation images, and 2 test images.

 Aborting file move. 

In [ ]:
from pathlib import Path

DATASET_NAME = "DeadTrees"
IMAGE_FOLDER = "dataset_rgb"
MASK_FOLDER = "dataset_binary"

IMAGE_PATH = Path(f"{DATASET_NAME}/{IMAGE_FOLDER}")
MASK_PATH = Path(f"{DATASET_NAME}/{MASK_FOLDER}")
TRAIN_FOLDER = f"{DATASET_NAME}/train"
VAL_FOLDER = f"{DATASET_NAME}/val"
TEST_FOLDER = f"{DATASET_NAME}/test"

In [ ]:
from pathlib import Path


In [20]:
import rasterio


def detect_non_border_images(image_list, percentage_threshold=0.1):
    blank_image_ids = []
    for path in image_list:
        image = rasterio.open(str(path)).read().transpose(1, 2, 0)
        ## Detecting Images with any number of white Pixels
        if (image.sum(axis=2) == 3 * 255).sum() / image[
            :, :, 0
        ].size <= percentage_threshold:
            blank_image_ids.append(Path(path).name)

    print("Number of Non-Border Images:", len(blank_image_ids))
    return blank_image_ids


def calculate_non_empty_masks(values, mask_path, s_id=False):
    valid_masks = []
    for value in values:
        if rasterio.open(mask_path / value).read().sum() > 0:
            valid_masks.append(value)
    print("Number of non-empty masks:", len(valid_masks))
    return valid_masks


In [ ]:
from sklearn.model_selection import train_test_split

train_size = 0.8
dev_size = 1 - train_size
test_size = 0.5

image_paths = sorted(IMAGE_GLOB.glob("*.tif"))
mask_paths = sorted(MASK_GLOB.glob("*.tif"))

non_border_image_names = detect_non_border_images(image_paths)
non_empty_mask_names = calculate_non_empty_masks(non_border_image_names, MASK_PATH)


(7224, 209, 209)

In [28]:
def deadtrees_split_images_and_masks(
    non_border_images_names,
    non_empty_mask_names,
    dev_size=0.2,
    test_size=0.5,
    random_state=42,
):

    _, dev_image_names = train_test_split(
        non_empty_mask_names, test_size=dev_size, random_state=42
    )

    train_image_names = list(set(non_border_image_names) - set(dev_image_names))
    val_image_names, test_image_names = train_test_split(
        dev_image_names, test_size=test_size, random_state=42
    )

    return train_image_names, val_image_names, test_image_names


In [31]:
dev_size = 1 - train_size
train_image_names, val_image_names, test_image_names = deadtrees_split_images_and_masks(
    non_border_image_names,
    non_empty_mask_names,
    dev_size=dev_size,
    test_size=test_size,
    random_state=42,
)

In [34]:
def create_image_paths(image_names, image_path):
    paths = []
    for image_name in image_names:
        paths.append(str(image_path / image_name))
    return paths


train_images = create_image_paths(train_image_names, IMAGE_PATH)
train_masks = create_image_paths(train_image_names, MASK_PATH)
val_images = create_image_paths(val_image_names, IMAGE_PATH)
val_masks = create_image_paths(val_image_names, MASK_PATH)
test_images = create_image_paths(test_image_names, IMAGE_PATH)
test_masks = create_image_paths(test_image_names, MASK_PATH)

In [36]:
from rich import print

print(
    f"[bold yellow] The process with create {len(train_images)} training images, {len(val_images)} validation images, and {len(test_images)} test images.[/bold yellow]"
)

 The process with create 7224 training images, 209 validation images, and 209 test images.